# Hyperparameter Tuning for Deep RL Inventory Optimization

**Objective:** Find optimal hyperparameters for DQN, Double DQN, and Dueling DQN

**Approach:** Grid search with cross-validation over key hyperparameters

---

In [ ]:
import numpy as np
import pandas as pd
import json
import itertools
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import environment and agents from main notebook
# Note: In practice, these would be imported from separate modules
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random
import os

# Set seeds
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

os.makedirs('tuning_results', exist_ok=True)

In [ ]:
# ===========================
# ENVIRONMENT (copied for standalone execution)
# ===========================

CAPACITY = 500
ORDER_LEVELS = list(range(0, 501, 50))
UNIT_PRICE = 14
UNIT_COST = 6
INV_HOLDING_COST = 0.5
CUSTOMER_LOST_COST = 4
WASTE_COST = UNIT_COST
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
EPISODE_DAYS = len(DAYS)

class RestaurantInventoryEnv:
    def __init__(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.action_space_size = len(ORDER_LEVELS)
        self.state_dim = 2
        self.reset()
    
    def reset(self):
        self.day_idx = 0
        self.inventory = np.random.randint(0, 150)
        return self._get_state()
    
    def _get_state(self):
        day_normalized = self.day_idx / (EPISODE_DAYS - 1)
        inventory_normalized = self.inventory / CAPACITY
        return np.array([day_normalized, inventory_normalized], dtype=np.float32)
    
    def step(self, action_idx):
        order_quantity = ORDER_LEVELS[action_idx]
        demand = np.random.poisson(250)
        
        inventory_after_order = self.inventory + order_quantity
        overflow = max(0, inventory_after_order - CAPACITY)
        inventory_after_order = min(inventory_after_order, CAPACITY)
        
        sales = min(inventory_after_order, demand)
        inventory_end = inventory_after_order - sales
        stockout = max(0, demand - sales)
        
        waste_units = overflow
        if self.day_idx == EPISODE_DAYS - 1:
            waste_units += inventory_end
            inventory_end = 0
        
        revenue = UNIT_PRICE * sales
        ordering_cost = UNIT_COST * order_quantity
        holding_cost = INV_HOLDING_COST * inventory_end
        stockout_cost = CUSTOMER_LOST_COST * stockout
        waste_cost = WASTE_COST * waste_units
        
        profit = revenue - ordering_cost - holding_cost - stockout_cost - waste_cost
        reward = profit / 1000.0
        
        self.inventory = inventory_end
        self.day_idx += 1
        done = self.day_idx >= EPISODE_DAYS
        
        next_state = self._get_state() if not done else np.zeros(self.state_dim)
        
        info = {'profit': profit}
        return next_state, reward, done, info
    
    def get_valid_actions(self):
        valid = []
        for i, order in enumerate(ORDER_LEVELS):
            if self.inventory + order <= CAPACITY:
                valid.append(i)
        return valid if valid else [0]

In [ ]:
# ===========================
# AGENT IMPLEMENTATIONS
# ===========================

class DQNNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims=[256, 256]):
        super(DQNNetwork, self).__init__()
        layers = []
        input_dim = state_dim
        for hidden_dim in hidden_dims:
            layers.extend([nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.LayerNorm(hidden_dim)])
            input_dim = hidden_dim
        layers.append(nn.Linear(input_dim, action_dim))
        self.network = nn.Sequential(*layers)
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
    
    def forward(self, state):
        return self.network(state)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(actions).to(device),
            torch.FloatTensor(rewards).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(dones).to(device)
        )
    
    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, 
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 buffer_size=50000, batch_size=64):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        self.policy_net = DQNNetwork(state_dim, action_dim).to(device)
        self.target_net = DQNNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_size)
        self.training_losses = []
    
    def select_action(self, state, valid_actions, explore=True):
        if explore and random.random() < self.epsilon:
            return random.choice(valid_actions)
        
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.policy_net(state_t).cpu().numpy().flatten()
            masked_q = np.full_like(q_values, -np.inf)
            masked_q[valid_actions] = q_values[valid_actions]
            return int(np.argmax(masked_q))
    
    def train_step(self):
        if len(self.memory) < self.batch_size:
            return
        
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.training_losses.append(loss.item())
    
    def update_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())
    
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)


class DoubleDQNAgent(DQNAgent):
    def train_step(self):
        if len(self.memory) < self.batch_size:
            return
        
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)
        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        
        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(1)
            next_q = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze()
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.training_losses.append(loss.item())


class DuelingDQNNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims=[256, 256]):
        super(DuelingDQNNetwork, self).__init__()
        shared_layers = []
        input_dim = state_dim
        for hidden_dim in hidden_dims[:-1]:
            shared_layers.extend([nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.LayerNorm(hidden_dim)])
            input_dim = hidden_dim
        self.feature_layer = nn.Sequential(*shared_layers)
        
        self.value_stream = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[-1]), nn.ReLU(), nn.Linear(hidden_dims[-1], 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[-1]), nn.ReLU(), nn.Linear(hidden_dims[-1], action_dim)
        )
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
    
    def forward(self, state):
        features = self.feature_layer(state)
        value = self.value_stream(features)
        advantages = self.advantage_stream(features)
        q_values = value + (advantages - advantages.mean(dim=1, keepdim=True))
        return q_values


class DuelingDQNAgent(DQNAgent):
    def __init__(self, state_dim, action_dim, **kwargs):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = kwargs.get('gamma', 0.99)
        self.batch_size = kwargs.get('batch_size', 64)
        self.epsilon = kwargs.get('epsilon_start', 1.0)
        self.epsilon_end = kwargs.get('epsilon_end', 0.05)
        self.epsilon_decay = kwargs.get('epsilon_decay', 0.995)
        
        self.policy_net = DuelingDQNNetwork(state_dim, action_dim).to(device)
        self.target_net = DuelingDQNNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=kwargs.get('lr', 1e-3))
        self.memory = ReplayBuffer(kwargs.get('buffer_size', 50000))
        self.training_losses = []

## Hyperparameter Search Space

In [ ]:
# ===========================
# HYPERPARAMETER GRID
# ===========================

PARAM_GRID = {
    'lr': [1e-4, 5e-4, 1e-3],
    'gamma': [0.95, 0.99],
    'epsilon_decay': [0.99, 0.995, 0.998],
    'batch_size': [32, 64],
    'buffer_size': [30000, 50000]
}

# Calculate total combinations
total_combinations = np.prod([len(v) for v in PARAM_GRID.values()])
print(f"Total hyperparameter combinations to test: {total_combinations}")
print(f"\nSearch space:")
for param, values in PARAM_GRID.items():
    print(f"  {param}: {values}")

In [ ]:
# ===========================
# TRAINING AND EVALUATION
# ===========================

def train_and_evaluate(agent_class, env, params, 
                      train_episodes=3000, eval_episodes=1000, 
                      target_update_freq=10):
    """
    Train agent and evaluate performance
    
    Returns: mean profit, std profit, training time
    """
    import time
    
    # Create agent
    agent = agent_class(
        state_dim=env.state_dim,
        action_dim=env.action_space_size,
        **params
    )
    
    # Training
    start_time = time.time()
    
    for episode in range(train_episodes):
        state = env.reset()
        done = False
        
        while not done:
            valid_actions = env.get_valid_actions()
            action = agent.select_action(state, valid_actions, explore=True)
            next_state, reward, done, info = env.step(action)
            agent.memory.push(state, action, reward, next_state, done)
            agent.train_step()
            state = next_state
        
        if episode % target_update_freq == 0:
            agent.update_target_network()
        
        agent.decay_epsilon()
    
    training_time = time.time() - start_time
    
    # Evaluation
    eval_profits = []
    for _ in range(eval_episodes):
        state = env.reset()
        done = False
        episode_profit = 0
        
        while not done:
            valid_actions = env.get_valid_actions()
            action = agent.select_action(state, valid_actions, explore=False)
            state, reward, done, info = env.step(action)
            episode_profit += info['profit']
        
        eval_profits.append(episode_profit)
    
    mean_profit = np.mean(eval_profits)
    std_profit = np.std(eval_profits)
    
    return mean_profit, std_profit, training_time


def grid_search(agent_class, agent_name, param_grid, n_seeds=3):
    """
    Perform grid search with multiple seeds
    
    Returns: DataFrame with results
    """
    results = []
    
    # Generate all parameter combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    param_combinations = list(itertools.product(*param_values))
    
    total_runs = len(param_combinations) * n_seeds
    print(f"\n{'='*80}")
    print(f"Grid Search for {agent_name}")
    print(f"{'='*80}")
    print(f"Total runs: {total_runs} ({len(param_combinations)} combinations × {n_seeds} seeds)")
    print(f"{'='*80}\n")
    
    with tqdm(total=total_runs, desc=f"{agent_name} Grid Search") as pbar:
        for combo in param_combinations:
            params = dict(zip(param_names, combo))
            
            # Run with multiple seeds for robustness
            seed_profits = []
            seed_stds = []
            seed_times = []
            
            for seed in range(n_seeds):
                env = RestaurantInventoryEnv(seed=seed)
                np.random.seed(seed)
                random.seed(seed)
                torch.manual_seed(seed)
                
                mean_profit, std_profit, train_time = train_and_evaluate(
                    agent_class, env, params
                )
                
                seed_profits.append(mean_profit)
                seed_stds.append(std_profit)
                seed_times.append(train_time)
                
                pbar.update(1)
            
            # Average across seeds
            result = params.copy()
            result['mean_profit'] = np.mean(seed_profits)
            result['std_profit'] = np.mean(seed_stds)
            result['profit_std_across_seeds'] = np.std(seed_profits)
            result['train_time'] = np.mean(seed_times)
            results.append(result)
    
    df = pd.DataFrame(results)
    df = df.sort_values('mean_profit', ascending=False).reset_index(drop=True)
    
    return df

## Run Grid Search for All Models

In [ ]:
# ===========================
# DQN TUNING
# ===========================

print("\n" + "="*80)
print("STARTING DQN HYPERPARAMETER TUNING")
print("="*80 + "\n")

dqn_results = grid_search(DQNAgent, "DQN", PARAM_GRID, n_seeds=3)

print("\nTop 5 DQN Configurations:")
print(dqn_results.head().to_string(index=False))

# Save results
dqn_results.to_csv('tuning_results/dqn_grid_search.csv', index=False)
print("\nDQN results saved to: tuning_results/dqn_grid_search.csv")

In [ ]:
# ===========================
# DOUBLE DQN TUNING
# ===========================

print("\n" + "="*80)
print("STARTING DOUBLE DQN HYPERPARAMETER TUNING")
print("="*80 + "\n")

ddqn_results = grid_search(DoubleDQNAgent, "Double DQN", PARAM_GRID, n_seeds=3)

print("\nTop 5 Double DQN Configurations:")
print(ddqn_results.head().to_string(index=False))

# Save results
ddqn_results.to_csv('tuning_results/ddqn_grid_search.csv', index=False)
print("\nDouble DQN results saved to: tuning_results/ddqn_grid_search.csv")

In [ ]:
# ===========================
# DUELING DQN TUNING
# ===========================

print("\n" + "="*80)
print("STARTING DUELING DQN HYPERPARAMETER TUNING")
print("="*80 + "\n")

dueling_results = grid_search(DuelingDQNAgent, "Dueling DQN", PARAM_GRID, n_seeds=3)

print("\nTop 5 Dueling DQN Configurations:")
print(dueling_results.head().to_string(index=False))

# Save results
dueling_results.to_csv('tuning_results/dueling_dqn_grid_search.csv', index=False)
print("\nDueling DQN results saved to: tuning_results/dueling_dqn_grid_search.csv")

## Extract Best Hyperparameters

In [ ]:
# ===========================
# EXTRACT BEST PARAMETERS
# ===========================

def extract_best_params(df, param_names):
    """Extract best hyperparameters from results DataFrame"""
    best_row = df.iloc[0]
    return {param: best_row[param] for param in param_names}

param_names = list(PARAM_GRID.keys())

best_params = {
    'DQN': extract_best_params(dqn_results, param_names),
    'DDQN': extract_best_params(ddqn_results, param_names),
    'DuelingDQN': extract_best_params(dueling_results, param_names)
}

print("\n" + "="*80)
print("BEST HYPERPARAMETERS")
print("="*80 + "\n")

for model_name, params in best_params.items():
    print(f"{model_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")
    print()

# Save to JSON
with open('hyperparameters_tuned.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print("\nBest parameters saved to: hyperparameters_tuned.json")
print("Use these parameters in the main training notebook!")

## Visualize Tuning Results

In [ ]:
# ===========================
# VISUALIZATION
# ===========================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Hyperparameter Tuning Results', fontsize=18, fontweight='bold')

all_results = [
    (dqn_results, 'DQN', '#1f77b4'),
    (ddqn_results, 'Double DQN', '#2ca02c'),
    (dueling_results, 'Dueling DQN', '#9467bd')
]

# Learning Rate Effect
ax = axes[0, 0]
for df, name, color in all_results:
    lr_grouped = df.groupby('lr')['mean_profit'].mean()
    ax.plot(lr_grouped.index, lr_grouped.values, marker='o', label=name, color=color, linewidth=2)
ax.set_xlabel('Learning Rate', fontsize=12)
ax.set_ylabel('Mean Profit ($)', fontsize=12)
ax.set_title('Effect of Learning Rate', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend()
ax.grid(alpha=0.3)

# Gamma Effect
ax = axes[0, 1]
for df, name, color in all_results:
    gamma_grouped = df.groupby('gamma')['mean_profit'].mean()
    ax.plot(gamma_grouped.index, gamma_grouped.values, marker='o', label=name, color=color, linewidth=2)
ax.set_xlabel('Discount Factor (γ)', fontsize=12)
ax.set_ylabel('Mean Profit ($)', fontsize=12)
ax.set_title('Effect of Discount Factor', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Epsilon Decay Effect
ax = axes[0, 2]
for df, name, color in all_results:
    eps_grouped = df.groupby('epsilon_decay')['mean_profit'].mean()
    ax.plot(eps_grouped.index, eps_grouped.values, marker='o', label=name, color=color, linewidth=2)
ax.set_xlabel('Epsilon Decay', fontsize=12)
ax.set_ylabel('Mean Profit ($)', fontsize=12)
ax.set_title('Effect of Epsilon Decay', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Batch Size Effect
ax = axes[1, 0]
for df, name, color in all_results:
    batch_grouped = df.groupby('batch_size')['mean_profit'].mean()
    ax.plot(batch_grouped.index, batch_grouped.values, marker='o', label=name, color=color, linewidth=2)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Mean Profit ($)', fontsize=12)
ax.set_title('Effect of Batch Size', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Buffer Size Effect
ax = axes[1, 1]
for df, name, color in all_results:
    buffer_grouped = df.groupby('buffer_size')['mean_profit'].mean()
    ax.plot(buffer_grouped.index, buffer_grouped.values, marker='o', label=name, color=color, linewidth=2)
ax.set_xlabel('Buffer Size', fontsize=12)
ax.set_ylabel('Mean Profit ($)', fontsize=12)
ax.set_title('Effect of Buffer Size', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Best Configuration Comparison
ax = axes[1, 2]
best_profits = [df.iloc[0]['mean_profit'] for df, _, _ in all_results]
best_stds = [df.iloc[0]['std_profit'] for df, _, _ in all_results]
model_names = [name for _, name, _ in all_results]
colors_list = [color for _, _, color in all_results]

x_pos = np.arange(len(model_names))
ax.bar(x_pos, best_profits, yerr=best_stds, color=colors_list, alpha=0.7, capsize=10)
ax.set_xticks(x_pos)
ax.set_xticklabels(model_names)
ax.set_ylabel('Mean Profit ($)', fontsize=12)
ax.set_title('Best Configuration Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (profit, std) in enumerate(zip(best_profits, best_stds)):
    ax.text(i, profit + std + 50, f'${profit:.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('tuning_results/hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Performance Summary
summary_data = []
for df, name, _ in all_results:
    best = df.iloc[0]
    summary_data.append({
        'Model': name,
        'Best Profit': f"${best['mean_profit']:.2f}",
        'Std': f"${best['std_profit']:.2f}",
        'LR': best['lr'],
        'Gamma': best['gamma'],
        'Eps Decay': best['epsilon_decay'],
        'Batch': int(best['batch_size']),
        'Buffer': int(best['buffer_size']),
        'Train Time': f"{best['train_time']:.1f}s"
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*120)
print("HYPERPARAMETER TUNING SUMMARY")
print("="*120)
print(summary_df.to_string(index=False))
print("="*120)

summary_df.to_csv('tuning_results/tuning_summary.csv', index=False)
print("\nSummary saved to: tuning_results/tuning_summary.csv")

## Conclusion

The tuned hyperparameters have been saved to `hyperparameters_tuned.json` and should be used in the main training notebook for optimal performance.